# Lab 3b: VDB Corpus Acquisition with FAISS
**Course:** CIS 531/731 | **Module 3: Obtain**

## Objective
Take the text output generated by the Whisper ASR pipeline, convert it into semantic embeddings using a HuggingFace Transformer, and index it in a local Vector Database (FAISS).

### DevContainer Setup Instructions
Your `verify_env.py` script from MP2 already confirmed the presence of `faiss` and `transformers`. If your DevContainer failed to build those, rebuild the container now. If you are running locally outside the container, you must run:
`!pip install faiss-cpu transformers sentence-transformers`

In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"FAISS Version: {faiss.__version__}")

/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


FAISS Version: 1.8.0


### 1. Initialize the Embedding Model
We use `all-MiniLM-L6-v2` as a lightweight, highly efficient sentence embedding model to encode our transcriptions into high-dimensional vectors.

In [2]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vector_dimension = embedder.get_sentence_embedding_dimension()
print(f"Model loaded. Embedding vector dimension: {vector_dimension}")

/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded. Embedding vector dimension: 384


### 2. Create the FAISS Index & Ingest Corpus
Here we simulate the ingestion of chunks coming from our live radio stream pipeline.

In [3]:
# Initialize a flat L2 (Euclidean distance) FAISS index
index = faiss.IndexFlatL2(vector_dimension)

# Simulated incoming ASR corpus chunks
corpus = [
    "The caller sounds really upset about their recent breakup.",
    "I love this Cage the Elephant track, it reminds me of summer.",
    "Sometimes I feel like our whole generation is just out of time."
]

# Encode the text into vectors
corpus_embeddings = embedder.encode(corpus)

# Add vectors to the VDB
index.add(np.array(corpus_embeddings).astype('float32'))
print(f"Successfully ingested {index.ntotal} vectors into FAISS.")

Successfully ingested 3 vectors into FAISS.


### 3. Query the VDB (Mapping Subtle Tropes)
We query the database for a semantic concept ("doomed youth") to see which transcribed chunk matches closest in latent space.

In [4]:
# 1. Bring in the Whisper output from Lab 3a
whisper_text = ""
print(f"Whisper Transcription to Ingest: '{whisper_text}'")

# 2. Embed the Whisper text
whisper_embedding = embedder.encode([whisper_text]).astype('float32')

# 3. Add to the existing FAISS index
index.add(whisper_embedding)
print(f"Successfully ingested Whisper chunk. Index now contains {index.ntotal} vectors.")

# 4. Append to corpus list for reverse-lookup later
full_corpus = corpus + [whisper_text]

Whisper Transcription to Ingest: ''
Successfully ingested Whisper chunk. Index now contains 4 vectors.


In [5]:
# 1. Write the FAISS index to disk
index_filename = "trope_vdb.index"
faiss.write_index(index, index_filename)
print(f"Index successfully saved to disk as {index_filename}")

# 2. Read the FAISS index back into memory
loaded_index = faiss.read_index(index_filename)
print(f"Index loaded from disk. Verified vector count: {loaded_index.ntotal}")

# 3. Define and embed the query
query = "doomed youth"
query_embedding = embedder.encode([query]).astype('float32')

# 4. Search the loaded index (k=1)
distances, indices = loaded_index.search(query_embedding, k=1)
matched_index = indices[0][0]

print("\n--- SEMANTIC SEARCH RESULTS ---")
print(f"Query: '{query}'")
print(f"Closest Match in VDB: '{full_corpus[matched_index]}'")
print(f"L2 Distance: {distances[0][0]:.4f}")

Index successfully saved to disk as trope_vdb.index
Index loaded from disk. Verified vector count: 4

--- SEMANTIC SEARCH RESULTS ---
Query: 'doomed youth'
Closest Match in VDB: 'Sometimes I feel like our whole generation is just out of time.'
L2 Distance: 1.2453
